In [ ]:
import pandas as pd
import numpy as np
from keras import callbacks
import tensorflow as tf
from datetime import datetime
from keras.models import Sequential
from keras.layers import Conv1D
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Input
from keras.layers import GlobalMaxPooling1D
from keras.layers import Concatenate
from keras import Model
from keras.layers import BatchNormalization
from keras.layers import MaxPooling1D
from keras.layers import GlobalAveragePooling1D


podaci = pd.read_csv("IMDB Dataset.csv", header=0)
vektori = "vectors.txt"

podaci['ocena'] = (podaci['sentiment'] == 'positive').astype(int)

deo_za_trening = 0.8
#%%
def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def load_word_vectors(filename):

    word_vectors = {}
    word2id = {}
    id2word = {}

    with open(filename, 'r', encoding='utf-8') as f:

        first_line = f.readline().strip()
        vocab_size, embedding_dim = map(int, first_line.split())

        for idx, line in enumerate(f):
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)

            word_vectors[word] = vector
            word2id[word] = idx
            id2word[idx] = word

    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, idx in word2id.items():
        embedding_matrix[idx] = word_vectors[word]

    return {
        'word_vectors': word_vectors,
        'embedding_matrix': embedding_matrix,
        'word2id': word2id,
        'id2word': id2word,
        'embedding_dim': embedding_dim,
        'vocab_size': vocab_size
    }

#%%
totalni_broj_tokena = 0
totalni_unc = 0
#%%
podaci['recenzija_tokeni'] = podaci['review'].apply(clean_and_tokenize)
vektorisane_reci = load_word_vectors(vektori)
#%%
def filter_and_replace_unknown_words(tokens, max_tokens = 400):
    tokeni_izlaz = []
    global totalni_unc,totalni_broj_tokena
    for id , token in enumerate(tokens, start = 0):
        if id >= max_tokens:
          break
        if token in vektorisane_reci['word2id'].keys():
            tokeni_izlaz.append(token)
        else:
            totalni_unc += 1
            tokeni_izlaz.append('UNK')
        totalni_broj_tokena += 1

    return tokeni_izlaz


podaci['filterisano'] = podaci['recenzija_tokeni'].apply(filter_and_replace_unknown_words)
print(f'Totalni broj tokena je {totalni_broj_tokena}, broj reci izvan vokabulara je {totalni_unc}')
print('To je ' + str((totalni_unc/totalni_broj_tokena)*100) + " posto")

Totalni broj tokena je 10112098, broj reci izvan vokabulara je 708537
To je 7.006824894299878 posto


In [ ]:
#%%

gener = np.random.default_rng(14)

podaci_pozitivni = podaci.loc[podaci['ocena'] == 1,:]
podaci_negativni = podaci.loc[podaci['ocena'] == 0,:]

podaci_pozitvni_trening = podaci_pozitivni.sample(frac = deo_za_trening, random_state = gener.bit_generator)
podaci_negativni_trening = podaci_negativni.sample(frac = deo_za_trening,  random_state = gener.bit_generator)

podaci_trening = pd.concat([podaci_pozitvni_trening, podaci_negativni_trening], ignore_index=True)[[
        "filterisano",
        "ocena"
    ]]
podaci_trening = podaci_trening.sample(frac = 1, random_state= 14).reset_index(drop = True)
print(f"U treningu je ukupno: {podaci_trening["ocena"].count()}, od toga pozitivno: {podaci_trening["ocena"].sum()}")


podaci_pozitvni_validacija = podaci_pozitivni.drop(index = podaci_pozitvni_trening.index)
podaci_negativni_validacija = podaci_negativni.drop(index = podaci_negativni_trening.index)

podaci_validacija = pd.concat([podaci_pozitvni_validacija, podaci_negativni_validacija], ignore_index=True)[[
        "filterisano",
        "ocena"
    ]]
podaci_validacija = podaci_validacija.sample(frac = 1, random_state= 14).reset_index(drop = True)
print(f"U validaciji je ukupno: {podaci_validacija["ocena"].count()}, od toga pozitivno: {podaci_validacija["ocena"].sum()}")


def generisi_batches(podaci, word_vectors, max_len = 400, batch_size = 32):
    x = []
    Y = []

    nula_vektor = np.zeros(200, dtype=np.float32)
    broj_redova = podaci.shape[0]

    while True:
      for start in range(0, broj_redova, batch_size):
            end = min(start + batch_size, broj_redova)
            batch_podaci = podaci.iloc[start:end]

            batch_x = []
            batch_y = []

            for _, red in batch_podaci.iterrows():
                tokeni = red['filterisano'][:max_len]
                ocena = red['ocena']

                matrica = [word_vectors.get(token, nula_vektor) for token in tokeni]

                if len(matrica) < max_len:
                    matrica.extend([nula_vektor] * (max_len - len(matrica)))

                batch_x.append(matrica)
                batch_y.append([1, 0] if ocena == 0 else [0, 1])


            yield np.array(batch_x, dtype=np.float32), np.array(batch_y, dtype=np.int8)

trening_generator = generisi_batches(podaci_trening, vektorisane_reci['word_vectors'], batch_size=32)
validacioni_generator = generisi_batches(podaci_validacija, vektorisane_reci['word_vectors'], batch_size=32)




U treningu je ukupno: 40000, od toga pozitivno: 20000
U validaciji je ukupno: 10000, od toga pozitivno: 5000


In [ ]:

cnn_model = Sequential()

cnn_model.add(Input(shape = (400, 200)))
cnn_model.add(BatchNormalization())
cnn_model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
cnn_model.add(GlobalAveragePooling1D())
cnn_model.add(Dense(64, activation='relu'))
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(2, activation='softmax'))



cnn_model.compile(optimizer="adam",
                  loss='binary_crossentropy',
                  metrics=[
                      'accuracy',
                      tf.keras.metrics.Precision(name='precision'),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.F1Score(name='f1_score', average='micro')
                  ])
steps_per_epoch = len(podaci_trening) // 32
validation_steps = len(podaci_validacija) // 32
trening = cnn_model.fit(
    trening_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validacioni_generator,
    validation_steps=validation_steps,
    verbose=1
)


In [ ]:
from keras.layers import AvgPool1D
cnn_model_2 = Sequential()

cnn_model_2.add(Input(shape = (400, 200)))
cnn_model_2.add(BatchNormalization())
cnn_model_2.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
cnn_model_2.add(AvgPool1D(pool_size=2))
cnn_model_2.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
cnn_model_2.add(GlobalAveragePooling1D())
cnn_model_2.add(Dense(64, activation='relu'))
cnn_model_2.add(Dropout(0.5))
cnn_model_2.add(Dense(2, activation='softmax'))



cnn_model_2.compile(optimizer="adam",
                  loss='binary_crossentropy',
                  metrics=[
                      'accuracy',
                      tf.keras.metrics.Precision(name='precision'),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.F1Score(name='f1_score', average='micro')
                  ])
steps_per_epoch = len(podaci_trening) // 32
validation_steps = len(podaci_validacija) // 32

trening_2 = cnn_model_2.fit(
    trening_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validacioni_generator,
    validation_steps=validation_steps,
    verbose=1
)



In [ ]:
from keras.layers import MaxPooling1D

cnn_model_3 = Sequential()

cnn_model_3.add(Input(shape = (400, 200)))
cnn_model_3.add(BatchNormalization())
cnn_model_3.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
cnn_model_3.add(MaxPooling1D(pool_size=2))
cnn_model_3.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
cnn_model_3.add(GlobalMaxPooling1D())
cnn_model_3.add(Dense(64, activation='relu'))
cnn_model_3.add(Dropout(0.5))
cnn_model_3.add(Dense(2, activation='softmax'))



cnn_model_3.compile(optimizer="adam",
                  loss='binary_crossentropy',
                  metrics=[
                      'accuracy',
                      tf.keras.metrics.Precision(name='precision'),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.F1Score(name='f1_score', average='micro')
                  ])
steps_per_epoch = len(podaci_trening) // 32
validation_steps = len(podaci_validacija) // 32

trening_3 = cnn_model_3.fit(
    trening_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validacioni_generator,
    validation_steps=validation_steps,
    verbose=1
)

In [ ]:
inputs = Input(shape=(400, 200))
normalizacija = BatchNormalization()(inputs)

conv2 = Conv1D(filters=128, kernel_size=2, activation='relu')(normalizacija)
conv3 = Conv1D(filters=128, kernel_size=3, activation='relu')(normalizacija)
conv4 = Conv1D(filters=128, kernel_size=4, activation='relu')(normalizacija)
conv5 = Conv1D(filters=128, kernel_size=5, activation='relu')(normalizacija)


pool2 = MaxPooling1D(pool_size=2)(conv2)
pool3 = MaxPooling1D(pool_size=2)(conv3)
pool4 = MaxPooling1D(pool_size=2)(conv4)
pool5 = MaxPooling1D(pool_size=2)(conv5)



gap2 = GlobalMaxPooling1D()(pool2)
gap3 = GlobalMaxPooling1D()(pool3)
gap4 = GlobalMaxPooling1D()(pool4)
gap5 = GlobalMaxPooling1D()(pool5)


concat = Concatenate()([gap2, gap3, gap4, gap5])

dense1 = tf.keras.layers.Dense(128, activation='relu')(concat)
dropout1 = tf.keras.layers.Dropout(0.5)(dense1)

outputs = Dense(2, activation='softmax')(dropout1)

cnn_model_4 = Model(inputs = inputs, outputs = outputs)

cnn_model_4.compile(optimizer="adam",
                  loss='binary_crossentropy',
                  metrics=[
                      'accuracy',
                      tf.keras.metrics.Precision(name='precision'),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.F1Score(name='f1_score', average='micro')
                  ])
steps_per_epoch = len(podaci_trening) // 32
validation_steps = len(podaci_validacija) // 32

trening_4 = cnn_model_4.fit(
    trening_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validacioni_generator,
    validation_steps=validation_steps,
    verbose=1
)

In [ ]:

inputs = Input(shape=(400, 200))
normalizacija = BatchNormalization()(inputs)

conv2 = Conv1D(filters=128, kernel_size=2, activation='relu')(normalizacija)
conv3 = Conv1D(filters=128, kernel_size=3, activation='relu')(normalizacija)
conv4 = Conv1D(filters=128, kernel_size=4, activation='relu')(normalizacija)
conv5 = Conv1D(filters=128, kernel_size=5, activation='relu')(normalizacija)

gap2 = GlobalMaxPooling1D()(conv2)
gap3 = GlobalMaxPooling1D()(conv3)
gap4 = GlobalMaxPooling1D()(conv4)
gap5 = GlobalMaxPooling1D()(conv5)


concat = Concatenate()([gap2, gap3, gap4, gap5])

dense1 = tf.keras.layers.Dense(128, activation='relu')(concat)
dropout1 = tf.keras.layers.Dropout(0.5)(dense1)

outputs = Dense(2, activation='softmax')(dropout1)

cnn_model_5 = Model(inputs = inputs, outputs = outputs)

cnn_model_5.compile(optimizer="adam",
                  loss='binary_crossentropy',
                  metrics=[
                      'accuracy',
                      tf.keras.metrics.Precision(name='precision'),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.F1Score(name='f1_score', average='micro')
                  ])
steps_per_epoch = len(podaci_trening) // 32
validation_steps = len(podaci_validacija) // 32

trening_5 = cnn_model_5.fit(
    trening_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=validacioni_generator,
    validation_steps=validation_steps,
    verbose=1
)



In [ ]:
def print_f(model):

  print(f"\naccuracy = {model.history['accuracy']}")
  print(f"val_accuracy = {model.history['val_accuracy']}")


  print(f"\nloss = {model.history['loss']}")
  print(f"val_loss = {model.history['val_loss']}")


  print(f"\nprecision = {model.history['precision']}")
  print(f"val_precision = {model.history['val_precision']}")


  print(f"\nrecall = {model.history['recall']}")
  print(f"val_recall = {model.history['val_recall']}")


  print(f"\nf1_score = {model.history['f1_score']}")
  print(f"val_f1_score = {model.history['val_f1_score']}")


  best_epoch = np.argmax(model.history['val_accuracy']) + 1
  print(f"\n{'='*40}")
  print(f"NAJBOLJA EPOH: {best_epoch}")
  print(f"best_val_accuracy = {max(model.history['val_accuracy']):.4f}")
  print(f"best_val_loss = {model.history['val_loss'][best_epoch-1]:.4f}")
  print(f"best_val_f1 = {model.history['val_f1_score'][best_epoch-1]:.4f}")
  print("="*80)

print_f(trening)
print_f(trening_2)
print_f(trening_3)
print_f(trening_4)
print_f(trening_5)

In [ ]:
from keras.utils import plot_model

plot_model(
    cnn_model_4 ,
    to_file='moj_model.png',
    show_shapes=False,
    show_layer_names=True,
    dpi=500,
    show_layer_activations=True
)